# Netflix and Disney+ Data Cleaning

This notebook cleans and combines the Netflix and Disney+ catalog datasets. The original files in `data/raw` remain unchanged, while the resulting title-level dataset will be saved in `data/processed`.

In [12]:
from pathlib import Path

import pandas as pd

In [13]:
RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [14]:
netflix_raw = pd.read_csv(
    RAW_DATA_DIR / "netflix_titles.csv"
)

disney_raw = pd.read_csv(
    RAW_DATA_DIR / "disney_plus_titles.csv"
)

In [15]:
print("Netflix:", netflix_raw.shape)
print("Disney+:", disney_raw.shape)

Netflix: (8807, 12)
Disney+: (1450, 12)


In [16]:
netflix_clean = netflix_raw.copy()
disney_clean = disney_raw.copy()

In [17]:
def standardize_column_names(df):
    cleaned_df = df.copy()

    cleaned_df.columns = (
        cleaned_df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )

    return cleaned_df

netflix_clean = standardize_column_names(netflix_clean)
disney_clean = standardize_column_names(disney_clean)

print(netflix_clean.columns.tolist())
print(disney_clean.columns.tolist())

assert netflix_clean.columns.tolist() == disney_clean.columns.tolist()

print("Column validation passed.")

['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']
['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']
Column validation passed.


In [18]:
def strip_text_values(df):
    cleaned_df = df.copy()

    text_columns = cleaned_df.select_dtypes(
        include=["object", "string"]
    ).columns

    for column in text_columns:
        cleaned_df[column] = cleaned_df[column].astype("string").str.strip()

    return cleaned_df

netflix_clean = strip_text_values(netflix_clean)
disney_clean = strip_text_values(disney_clean)

In [19]:
misplaced_duration_mask = netflix_clean["rating"].str.fullmatch(
    r"\d+\s*min",
    na=False
)

netflix_clean.loc[
    misplaced_duration_mask,
    ["show_id", "title", "rating", "duration"]
]

netflix_clean.loc[
    misplaced_duration_mask,
    "duration"
] = netflix_clean.loc[
    misplaced_duration_mask,
    "rating"
]

netflix_clean.loc[
    misplaced_duration_mask,
    "rating"
] = pd.NA

remaining_misplaced_values = netflix_clean["rating"].str.fullmatch(
    r"\d+\s*min",
    na=False
).sum()

print(
    "Duration values remaining in rating:",
    remaining_misplaced_values
)

print(
    "Netflix missing durations:",
    netflix_clean["duration"].isna().sum()
)

print(
    "Netflix missing ratings:",
    netflix_clean["rating"].isna().sum()
)

Duration values remaining in rating: 0
Netflix missing durations: 0
Netflix missing ratings: 7


In [20]:
netflix_clean["platform"] = "Netflix"
disney_clean["platform"] = "Disney+"

netflix_clean["content_id"] = (
    "netflix_" + netflix_clean["show_id"]
)

disney_clean["content_id"] = (
    "disney_plus_" + disney_clean["show_id"]
)

print(
    "Netflix duplicate content IDs:",
    netflix_clean["content_id"].duplicated().sum()
)

print(
    "Disney+ duplicate content IDs:",
    disney_clean["content_id"].duplicated().sum()
)


Netflix duplicate content IDs: 0
Disney+ duplicate content IDs: 0


## Stage 2: Transform Dates, Durations, and Missing Values

This stage converts dates into a consistent datetime format, separates duration numbers from their units, and labels missing categorical information as `Unknown`. Missing dates will remain unavailable because estimating when a title was added would introduce unsupported information.

The two datasets will then be combined into one title-level dataset. Multi-value fields such as country and genre will remain intact at this stage so that every row continues to represent one title.

In [22]:
print(
    "Netflix missing dates before conversion:",
    netflix_clean["date_added"].isna().sum()
)

print(
    "Disney+ missing dates before conversion:",
    disney_clean["date_added"].isna().sum()
)

Netflix missing dates before conversion: 10
Disney+ missing dates before conversion: 3


In [24]:
def convert_date_added(df):
    cleaned_df = df.copy()

    cleaned_df["date_added"] = pd.to_datetime(
        cleaned_df["date_added"],
        format="%B %d, %Y",
        errors="coerce"
    )

    return cleaned_df

netflix_clean = convert_date_added(netflix_clean)
disney_clean = convert_date_added(disney_clean)

print(
    "Netflix date type:",
    netflix_clean["date_added"].dtype
)

print(
    "Disney+ date type:",
    disney_clean["date_added"].dtype
)

print(
    "Netflix missing dates after conversion:",
    netflix_clean["date_added"].isna().sum()
)

print(
    "Disney+ missing dates after conversion:",
    disney_clean["date_added"].isna().sum()
)

Netflix date type: datetime64[us]
Disney+ date type: datetime64[us]
Netflix missing dates after conversion: 10
Disney+ missing dates after conversion: 3


In [25]:
def separate_duration(df):
    cleaned_df = df.copy()

    duration_parts = cleaned_df["duration"].str.extract(
        r"(?P<duration_value>\d+)\s*(?P<duration_unit>min|Season|Seasons)"
    )

    cleaned_df["duration_value"] = pd.to_numeric(
        duration_parts["duration_value"],
        errors="coerce"
    ).astype("Int64")

    cleaned_df["duration_unit"] = (
        duration_parts["duration_unit"]
        .str.lower()
        .replace({
            "min": "Minutes",
            "season": "Seasons",
            "seasons": "Seasons"
        })
    )

    return cleaned_df

netflix_clean = separate_duration(netflix_clean)
disney_clean = separate_duration(disney_clean)

display(
    netflix_clean[
        ["type", "title", "duration", "duration_value", "duration_unit"]
    ].head()
)

display(
    disney_clean[
        ["type", "title", "duration", "duration_value", "duration_unit"]
    ].head()
)

,type,title,duration,duration_value,duration_unit
0,Movie,Dick Johnson Is Dead,90 min,90,Minutes
1,TV Show,Blood & Water,2 Seasons,2,Seasons
2,TV Show,Ganglands,1 Season,1,Seasons
3,TV Show,Jailbirds New Orleans,1 Season,1,Seasons
4,TV Show,Kota Factory,2 Seasons,2,Seasons


,type,title,duration,duration_value,duration_unit
0,Movie,Duck the Halls: A Mickey Mouse Christmas Special,23 min,23,Minutes
1,Movie,Ernest Saves Christmas,91 min,91,Minutes
2,Movie,Ice Age: A Mammoth Christmas,23 min,23,Minutes
3,Movie,The Queen Family Singalong,41 min,41,Minutes
4,TV Show,The Beatles: Get Back,1 Season,1,Seasons


In [26]:
def duration_validation(df, platform):
    failed_parsing = df[
        df["duration"].notna()
        & (
            df["duration_value"].isna()
            | df["duration_unit"].isna()
        )
    ]

    print(f"{platform} failed duration records: {len(failed_parsing)}")

    return failed_parsing

netflix_duration_errors = duration_validation(
    netflix_clean,
    "Netflix"
)

disney_duration_errors = duration_validation(
    disney_clean,
    "Disney+"
)

Netflix failed duration records: 0
Disney+ failed duration records: 0


In [27]:
def find_duration_type_mismatches(df):
    movie_mismatches = df[
        (df["type"] == "Movie")
        & (df["duration_unit"] != "Minutes")
    ]

    show_mismatches = df[
        (df["type"] == "TV Show")
        & (df["duration_unit"] != "Seasons")
    ]

    return pd.concat(
        [movie_mismatches, show_mismatches],
        ignore_index=True
    )

netflix_duration_mismatches = find_duration_type_mismatches(
    netflix_clean
)

disney_duration_mismatches = find_duration_type_mismatches(
    disney_clean
)

print(
    "Netflix duration/type mismatches:",
    len(netflix_duration_mismatches)
)

print(
    "Disney+ duration/type mismatches:",
    len(disney_duration_mismatches)
)


Netflix duration/type mismatches: 0
Disney+ duration/type mismatches: 0


In [29]:
CATEGORICAL_MISSING_COLUMNS = [
    "director",
    "cast",
    "country",
    "rating"
]

def label_missing_categories(df):
    cleaned_df = df.copy()

    for column in CATEGORICAL_MISSING_COLUMNS:
        cleaned_df[column] = cleaned_df[column].fillna("Unknown")

    return cleaned_df

netflix_clean = label_missing_categories(netflix_clean)
disney_clean = label_missing_categories(disney_clean)

print("Netflix remaining categorical missing values:")
display(
    netflix_clean[CATEGORICAL_MISSING_COLUMNS]
    .isna()
    .sum()
)

print("Disney+ remaining categorical missing values:")
display(
    disney_clean[CATEGORICAL_MISSING_COLUMNS]
    .isna()
    .sum()
)

print("Netflix Unknown values:")
display(
    netflix_clean[CATEGORICAL_MISSING_COLUMNS]
    .eq("Unknown")
    .sum()
)

print("Disney+ Unknown values:")
display(
    disney_clean[CATEGORICAL_MISSING_COLUMNS]
    .eq("Unknown")
    .sum()
)


Netflix remaining categorical missing values:


director    0
cast        0
country     0
rating      0
dtype: int64

Disney+ remaining categorical missing values:


director    0
cast        0
country     0
rating      0
dtype: int64

Netflix Unknown values:


director    2634
cast         825
country      831
rating         7
dtype: Int64

Disney+ Unknown values:


director    473
cast        190
country     219
rating        3
dtype: Int64

In [30]:
streaming_titles_clean = pd.concat(
    [netflix_clean, disney_clean],
    ignore_index=True
)

COLUMN_ORDER = [
    "content_id",
    "platform",
    "show_id",
    "type",
    "title",
    "director",
    "cast",
    "country",
    "date_added",
    "release_year",
    "rating",
    "duration",
    "duration_value",
    "duration_unit",
    "listed_in",
    "description"
]

streaming_titles_clean = streaming_titles_clean[COLUMN_ORDER]

print(
    "Combined dataset shape:",
    streaming_titles_clean.shape
)

print("\nRecords by platform:")
print(
    streaming_titles_clean["platform"].value_counts()
)

print(
    "\nDuplicate content IDs:",
    streaming_titles_clean["content_id"].duplicated().sum()
)

print(
    "Fully duplicated rows:",
    streaming_titles_clean.duplicated().sum()
)

Combined dataset shape: (10257, 16)

Records by platform:
platform
Netflix    8807
Disney+    1450
Name: count, dtype: int64

Duplicate content IDs: 0
Fully duplicated rows: 0


In [32]:
content_type_summary = pd.crosstab(
    streaming_titles_clean["platform"],
    streaming_titles_clean["type"]
)

display(content_type_summary)

remaining_missing_values = (
    streaming_titles_clean
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    remaining_missing_values[
        remaining_missing_values > 0
    ]
)

type,Movie,TV Show
platform,,
Disney+,1052,398
Netflix,6131,2676


date_added    13
dtype: int64

In [33]:
CLEANED_DATA_PATH = (
    PROCESSED_DATA_DIR / "streaming_titles_clean.csv"
)

streaming_titles_clean.to_csv(
    CLEANED_DATA_PATH,
    index=False
)

print(f"Cleaned dataset saved to: {CLEANED_DATA_PATH}")
print("File exists:", CLEANED_DATA_PATH.exists())

Cleaned dataset saved to: ..\data\processed\streaming_titles_clean.csv
File exists: True
